In [ ]:
# Cell 1: Cohort summary

import pandas as pd
import scanpy as sc
import os

from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")

# Load all datasets
datasets = {
    'Walesky': 'walesky_apap_control_raw.h5ad',
    'Umbaugh_2h': 'umbaugh_2h_filtered.h5ad',
    'Ben-Moshe': 'benmoshe_hepatocytes.h5ad',
    'Umbaugh_48h72h': 'umbaugh_48h72h_hepatocytes.h5ad',
    'Matchett_Human': 'matchett_human_hepatocytes.h5ad'
}

summary_data = []

for name, filename in datasets.items():
    filepath = os.path.join(data_processed, filename)
    print(f"Loading {name}...")
    adata = sc.read_h5ad(filepath)
    
    # Calculate QC metrics if not present
    if 'n_genes_by_counts' not in adata.obs.columns:
        print(f"  Calculating QC metrics for {name}...")
        sc.pp.calculate_qc_metrics(adata, inplace=True)
    
    # Determine species
    species = 'Human' if 'Human' in name else 'Mouse'
    
    # Platform
    platform_map = {
        'Walesky': 'Seq-Well',
        'Umbaugh_2h': '10x Chromium',
        'Ben-Moshe': '10x Chromium',
        'Umbaugh_48h72h': '10x Chromium',
        'Matchett_Human': '10x Chromium (snRNA-seq)'
    }
    
    # Accession
    accession_map = {
        'Walesky': 'GSE136679',
        'Umbaugh_2h': 'GSE200771',
        'Ben-Moshe': 'Zenodo 6035873',
        'Umbaugh_48h72h': 'GSE255834',
        'Matchett_Human': 'GSE223581'
    }
    
    summary_data.append({
        'Dataset': name,
        'Accession': accession_map[name],
        'Species': species,
        'Platform': platform_map[name],
        'Total_Cells': adata.n_obs,
        'Total_Genes': adata.n_vars,
        'Median_Genes_Cell': int(adata.obs['n_genes_by_counts'].median()),
        'Median_UMI_Cell': int(adata.obs['total_counts'].median()),
        'File_Size_MB': round(os.path.getsize(filepath) / 1e6, 1)
    })

master_df = pd.DataFrame(summary_data)

print("\n" + "="*100)
print("WEEK 1 MASTER SUMMARY - ALL DATASETS")
print("="*100)
print(master_df.to_string(index=False))
print("="*100)

print(f"\nTOTAL CELLS LOADED: {master_df['Total_Cells'].sum():,}")
print(f"AVERAGE QUALITY (median genes/cell): {master_df['Median_Genes_Cell'].mean():.0f}")
print(f"TOTAL DISK SPACE: {master_df['File_Size_MB'].sum():.1f} MB")

# Save
master_summary_file = os.path.join(data_processed, "week1_master_summary.csv")
master_df.to_csv(master_summary_file, index=False)
print(f"\n✅ Master summary saved: {master_summary_file}")